# 1. Setup

### 1.1 Install deps & packages

In [2]:
%pip install numpy pandas matplotlib kagglehub
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import kagglehub
import shutil
import os

You should consider upgrading via the '/Users/Hita/Desktop/Transaction-fraud-detection/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


/Users/Hita/Desktop/Transaction-fraud-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Download dataset

In [27]:
try:
    os.mkdir('original_dataset')
    kagglehub.dataset_download("zahranusratt/banking-fraud-detection-dataset", output_dir='original_dataset')
except FileExistsError:
    pass

# Remove uesless metadata
shutil.rmtree('original_dataset/.complete/', ignore_errors=True)

print('Dataset downloaded')

Dataset downloaded


# 2. Data pipeline

### 2.1 Load csv into dataframe

In [5]:
fraud_data = pd.read_csv('original_dataset/bank_fraud.csv')

print(f"Shape\n{fraud_data.shape}\n")
print(f"Head\n{fraud_data.head()}")
print("\nColumns")
for column_number, column_name in enumerate(fraud_data.columns, start=1):
    print(column_number, column_name)

Shape
(1000000, 26)

Head
  transaction_id   customer_id transaction_date transaction_time  hour_of_day  \
0  TXN0000000001  CUST00121959       2023-08-17         21:13:00           21   
1  TXN0000000002  CUST00146868       2024-02-06         05:16:00            5   
2  TXN0000000003  CUST00131933       2024-06-28         12:15:00           12   
3  TXN0000000004  CUST00103695       2023-03-16         02:53:00            2   
4  TXN0000000005  CUST00119880       2024-07-12         12:39:00           12   

   is_weekend  is_night_transaction country       city merchant_category  ...  \
0           0                     0     USA     London           Grocery  ...   
1           0                     1      UK   New York        Healthcare  ...   
2           0                     0  Canada      Delhi           Grocery  ...   
3           0                     1  France      Tokyo         Utilities  ...   
4           0                     0  Canada  Melbourne          Clothing  ...   



In [6]:
fraud_data_original = fraud_data.copy(deep=True)

print("Records:", fraud_data_original.shape[0])
print("Columns:", fraud_data_original.shape[1])

print("\nData-type counts:")
print(fraud_data_original.dtypes.value_counts())

Records: 1000000
Columns: 26

Data-type counts:
int64      11
object     10
float64     5
Name: count, dtype: int64


In [7]:
summary = pd.DataFrame({
    'dtype': fraud_data_original.dtypes,
    'sample_values': [fraud_data_original[col].dropna().unique()[:3] for col in fraud_data_original.columns]
})
print(summary.to_string())

                            dtype                                         sample_values
transaction_id             object         [TXN0000000001, TXN0000000002, TXN0000000003]
customer_id                object            [CUST00121959, CUST00146868, CUST00131933]
transaction_date           object                  [2023-08-17, 2024-02-06, 2024-06-28]
transaction_time           object                        [21:13:00, 05:16:00, 12:15:00]
hour_of_day                 int64                                           [21, 5, 12]
is_weekend                  int64                                                [0, 1]
is_night_transaction        int64                                                [0, 1]
country                    object                                     [USA, UK, Canada]
city                       object                             [London, New York, Delhi]
merchant_category          object                      [Grocery, Healthcare, Utilities]
payment_method             objec

2.  Numeric ranges, missing values, duplicates, and inconsistent entries

In [31]:


print("\n" + "="*60)
print("NUMERIC RANGES")
print("="*60)
print(fraud_data_original.describe())

print("\n" + "="*60)
print("UNIQUE VALUES (categorical columns)")
print("="*60)
for col in fraud_data_original.select_dtypes(include="object").columns:
    print(f"\n{col}: {fraud_data_original[col].nunique()} unique values")
    print(fraud_data_original[col].unique()[:15])

print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
missing = fraud_data_original.isna().sum()
missing_pct = (missing / len(fraud_data_original) * 100).round(2)
missing_summary = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(missing_summary[missing_summary["missing"] > 0].sort_values("missing", ascending=False))

print("\n" + "="*60)
print("DUPLICATE ROWS")
print("="*60)
print("Full duplicate rows:", fraud_data_original.duplicated().sum())
if "transaction_id" in fraud_data_original.columns:
    print("Duplicate transaction_id:", fraud_data_original["transaction_id"].duplicated().sum())


NUMERIC RANGES
       Transaction_ID   Customer_ID  Transaction_Amount (in Million)  \
count    49997.000000  49990.000000                     49991.000000   
mean    550400.968898  54869.720744                         4.999880   
std     259677.602349  26052.824933                         2.582025   
min     100043.000000  10005.000000                         1.000000   
25%     324445.000000  32259.250000                         3.000000   
50%     552115.000000  54720.500000                         5.000000   
75%     775942.000000  77542.000000                         7.000000   
max     999992.000000  99996.000000                         9.000000   

        Merchant_ID  Distance_From_Home      Device_ID  \
count  49993.000000        49998.000000   49991.000000   
mean   54951.375913          300.098564  552563.600088   
std    25983.342481          172.848263  260186.451027   
min    10001.000000            1.000000  100053.000000   
25%    32545.000000          150.000000  3276

# 3. Data Cleansing and Transformation

### 3.1 Dropping columns

Looking at the dataset, we can see the transaction ID is unique to every row, therefore it won't be useful in deriving any value.
We can use the customer contry to identify if the transaction took place in a high risk country, for this reason we can drop the city as it's too spesific.

In [32]:
fraud_data = fraud_data.drop(columns=['transaction_id', 'country', 'city'])

fraud_data.head()

KeyError: "['transaction_id', 'country', 'city'] not found in axis"

### 3.2 Normalising data

Since a lot of these columns are categories we can convert them to a number using a dict

In [ ]:
# A list of categorical columns
categorical_columns = ['merchant_category', 'payment_method', 'device_type', 'fraud_type']

# Create a dict of the enum values for each category
category_values = {}

for category in categorical_columns:
    category_values[category] = fraud_data[category].unique().tolist()

# Apply the normalisation
for i in range(fraud_data.shape[0]):
    for key in category_values:
        fraud_data.at[i, key] = category_values[key].index(fraud_data.loc[i][key])
    
fraud_data.head()

### 3.3 Creating a new feature for identifying high risk transactions

We can create a new feature called `high_risk` for what is roughly a high risk transaction, this would be defined by
- `account_age_years` <= 1
- `time_since_last_txn_hrs` <= 1
- `is_international` == 1
- `pin_changed_recently` == 1
- `transaction_amount` >= 100

In [ ]:
fraud_data['high_risk'] = np.where(
    (fraud_data['account_age_years'] <= 1) &
    (fraud_data['time_since_last_txn_hrs'] <= 1) &
    (fraud_data['is_international'] == 1) &
    (fraud_data['transaction_amount'] >= 100),
    1, 0)

filtered_df = fraud_data[fraud_data['high_risk'] == 1]

print(f"Found {filtered_df.shape[0]} high risk transactions")

fraud_data['high_risk'].describe()